# LLM clients

This notebook demonstrates the usage of Kaval.AI LLM clients.

In [1]:
# Load dotenv
import dotenv

dotenv.load_dotenv("../.env")

True

## Basic prompting and text generation

In [2]:
from kavalai.llm_clients.v2.gemini_client import GeminiClient
from kavalai.llm_clients.v2.openai_client import OpenAIClient

gemini_client = GeminiClient(model="gemini-3.1-flash-lite")
openai_client = OpenAIClient(model="gpt-5.4-mini")

/home/timo/projects/kaval.ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
gemini_result = await gemini_client.prompt("Start a conversation with OpenAI, say who you are. Max 50 words.")
print(f"Gemini: {gemini_result}\n")

openai_result = await openai_client.prompt(f"Gemini: f{gemini_result}")
print(f"OpenAI: {openai_result}\n")

Gemini: Hello, OpenAI. I am an AI assistant designed to provide helpful, accurate, and creative information. I am reaching out to initiate a dialogue, explore potential synergies, and learn from your advanced language models. I look forward to our interaction.

OpenAI: Hello! Nice to meet you. I’m happy to collaborate, compare approaches, or help with any task you’d like to explore. If you have a question, a prompt, or a project in mind, send it over and I’ll do my best to assist.



## Structured outputs

In [4]:
from pydantic import BaseModel

html = """
<table>
  <tr>
    <th>Company</th>
    <th>Contact</th>
    <th>Location</th>
  </tr>
  <tr>
    <td>Kaval.AI</td>
    <td>Timo Petmanson</td>
    <td>Estonia</td>
  </tr>
  <tr>
    <td>Intergalactic Government</td>
    <td>😼</td>
    <td>Jupiter</td>
  </tr>
</table>
"""


class Company(BaseModel):
    name: str
    contact_info: str
    country: str


class ParseResult(BaseModel):
    companies: list[Company]


companies = await gemini_client.prompt(f"Parse company data from HTML\n{html}",
                                       response_model=ParseResult)

print(companies)


companies=[Company(name='Kaval.AI', contact_info='Timo Petmanson', country='Estonia'), Company(name='Intergalactic Government', contact_info='😼', country='Jupiter')]


## Streaming

In [5]:
streamer = await openai_client.stream_prompt("Count to 3. Separate values by commas.")
async for chunk in streamer:
    print(chunk)


type='partial' name='response' value='1'
type='partial' name='response' value='1,'
type='partial' name='response' value='1, '
type='partial' name='response' value='1, 2'
type='partial' name='response' value='1, 2,'
type='partial' name='response' value='1, 2, '
type='partial' name='response' value='1, 2, 3'
type='complete' name='response' value='1, 2, 3'


Streaming structured data always returns valid JSON for partial responses.

In [6]:
class UserProfile(BaseModel):
    name: str
    birthdate: str
    skills: list[str]
    hobbies: list[str]


streamer = await gemini_client.stream_prompt("Generate a mock user profile",
                                             response_model=UserProfile)
async for chunk in streamer:
    print(chunk)

type='partial' name='response' value='{"name": "Elena"}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14"}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"]}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"], "hobbies": ["Hiking", "Photography", "Cooking"]}'
type='complete' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"], "hobbies": ["Hiking", "Photography", "Cooking"]}'
